# TruthGuard — Notebook 1/5
## Étape 1 : Installation, Chargement des données & EDA Labels

## 1. Installation & Imports

In [ ]:
%pip install nltk wordcloud sentence-transformers shap langdetect --break-system-packages -q

In [ ]:
import os, re, string, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import nltk

resources = [
    'punkt',
    'stopwords',
    'wordnet',
    'omw-1.4',
    'averaged_perceptron_tagger'
]

for resource in resources:
    try:
        nltk.download(resource)
    except:
        print(f"Erreur téléchargement : {resource}")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

DATA_DIR = Path(r"C:\Users\mouss\Downloads\projet\dataset")
OUT_DIR  = Path(r"C:\Users\mouss\Downloads\projet\output")
SEED = 42

## 2. Chargement et fusion des données

In [ ]:
COLS_TO_DROP = [ "url", "top_img", "authors",
                 "movies", "images", "canonical_link", "meta_data"]

def load_and_prepare(filepath: Path, label: int, source_name: str,
                     extra_drop: list = None) -> pd.DataFrame:
    """Charge un CSV, crée la colonne 'statement' et ajoute les métadonnées."""
    df = pd.read_csv(filepath, engine='python', on_bad_lines='warn')
    cols_present = [c for c in (COLS_TO_DROP + (extra_drop or [])) if c in df.columns]
    df.drop(columns=cols_present, inplace=True)
    df["label"]      = label
    df["statement"]  = df["title"].fillna("") + " " + df["text"].fillna("")
    df["source_df"]  = source_name
    df.drop(columns=[c for c in ["title", "text"] if c in df.columns], inplace=True)
    return df

df_fnn = pd.concat([
    load_and_prepare(DATA_DIR / "BuzzFeed_fake_news_content.csv",   0, "buzzfeed"),
    load_and_prepare(DATA_DIR / "BuzzFeed_real_news_content.csv",   1, "buzzfeed"),
    load_and_prepare(DATA_DIR / "PolitiFact_fake_news_content.csv", 0, "politifact"),
    load_and_prepare(DATA_DIR / "PolitiFact_real_news_content.csv", 1, "politifact"),
], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

df_raw = df_fnn.copy()

print(f"FakeNewsNet (BuzzFeed + PolitiFact) : {df_raw.shape[0]:>6} lignes")
df_raw.head(3)

## 3. EDA — Vue d'ensemble du dataset

In [ ]:
print("═" * 50)
print(" APERÇU GÉNÉRAL DU DATASET BRUT")
print("═" * 50)
print(f"\n• Dimensions    : {df_raw.shape}")
print(f"• Colonnes      : {list(df_raw.columns)}")
print(f"• Types         :\n{df_raw.dtypes}")
print(f"\n• Valeurs nulles par colonne :")
print(df_raw.isnull().sum())
print(f"\n• Doublons (statement identique) : {df_raw.duplicated(subset='statement').sum()}")

In [ ]:
df_raw["char_count"]  = df_raw["statement"].str.len()
df_raw["word_count"]  = df_raw["statement"].str.split().str.len()
df_raw["sent_count"]  = df_raw["statement"].apply(lambda x: len(sent_tokenize(str(x))))
df_raw["avg_word_len"]= df_raw["statement"].apply(
    lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
)

print(df_raw[["char_count", "word_count", "sent_count", "avg_word_len"]].describe().round(2))

## 4. EDA — Analyse des labels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

label_counts = df_raw["label"].value_counts().sort_index()
label_names  = ["Fake (0)", "Real (1)"]
colors       = ["#e74c3c", "#2980b9"]
bars = axes[0].bar(label_names, label_counts.values, color=colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, label_counts.values):
    pct = val / label_counts.sum() * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f"{val:,}\n({pct:.1f}%)", ha="center", fontsize=10, fontweight="bold")
axes[0].set_title("Distribution globale des classes", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Nombre d'articles")
axes[0].set_ylim(0, label_counts.max() * 1.15)

src_label = df_raw.groupby(["source_df", "label"]).size().unstack(fill_value=0)
src_label.columns = ["Fake", "Real"]
src_label.plot(kind="bar", ax=axes[1], color=["#e74c3c", "#2980b9"],
               edgecolor="white", linewidth=1)
axes[1].set_title("Répartition Fake/Real par source", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Source")
axes[1].set_ylabel("Nombre d'articles")
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title="Label")

src_ratio = src_label["Fake"] / (src_label["Real"] + 1e-9)
src_ratio.plot(kind="barh", ax=axes[2], color="#8e44ad", edgecolor="white")
axes[2].axvline(x=1.0, color="black", linestyle="--", linewidth=1.5, label="Équilibre (ratio=1)")
axes[2].set_title("Ratio Fake/Real par source", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Ratio (>1 = plus de fake)")
axes[2].legend()

plt.suptitle("Analyse des labels", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

ratio = label_counts.min() / label_counts.max()
print(f"\n  Ratio déséquilibre (min/max) : {ratio:.3f}")
if ratio < 0.7:
    print("   → Déséquilibre détecté : utiliser class_weight='balanced' ou sur-/sous-échantillonnage.")
else:
    print("   → Dataset relativement équilibré.")